# 🛠️ Interactive Fuzzy Logic Designer App

## 🎯 **Complete Fuzzy Inference System Builder**

This interactive tool allows you to:

### ✨ **Features:**
- **🎛️ Design Components**: Create custom membership functions and rules
- **📊 Visual Interface**: Interactive parameter adjustment
- **🔍 Step-by-Step Inference**: Visualize each stage of the fuzzy process
- **📈 Real-time Results**: Instant crisp output calculation
- **💾 Save/Load**: Export and import your fuzzy systems

### 🔧 **Process Steps:**
1. **Define Variables**: Input and output universes
2. **Create Membership Functions**: Choose types and parameters
3. **Build Rules**: Define IF-THEN logic
4. **Test Inputs**: Get crisp outputs with visualization
5. **Analyze Results**: Complete inference process breakdown

Let's build your custom fuzzy logic system! 🚀

In [ ]:
# 📦 Setup and Dependencies
!pip install scikit-fuzzy ipywidgets matplotlib numpy --quiet

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings
warnings.filterwarnings('ignore')

try:
    import skfuzzy as fuzz
    from skfuzzy import control as ctrl
    print("✅ All libraries loaded successfully!")
except ImportError:
    print("⚠️ Installing scikit-fuzzy...")
    !pip install scikit-fuzzy
    import skfuzzy as fuzz
    from skfuzzy import control as ctrl
    print("✅ All libraries loaded successfully!")

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("🎨 Visualization setup complete!")
print("🧠 Ready to build fuzzy systems!")

In [ ]:
class FuzzySystemDesigner:
    """
    Interactive Fuzzy Logic System Designer
    
    A comprehensive tool for creating, testing, and visualizing
    fuzzy inference systems with real-time interaction.
    """
    
    def __init__(self):
        self.variables = {}
        self.membership_functions = {}
        self.rules = []
        self.control_system = None
        self.simulation = None
        
        # Available membership function types
        self.mf_types = {
            'triangular': 'Triangular (3 parameters: a, b, c)',
            'trapezoidal': 'Trapezoidal (4 parameters: a, b, c, d)',
            'gaussian': 'Gaussian (2 parameters: mean, sigma)',
            'sigmoid': 'Sigmoid (2 parameters: a, c)'
        }
        
        print("🎛️ Fuzzy System Designer initialized!")
    
    def create_variable(self, name, var_type, universe_range, labels):
        """
        Create a fuzzy variable (input or output)
        
        Parameters:
        -----------
        name : str
            Variable name
        var_type : str
            'input' or 'output'
        universe_range : tuple
            (min, max) values
        labels : list
            Linguistic labels for the variable
        """
        universe = np.arange(universe_range[0], universe_range[1] + 0.1, 0.1)
        
        if var_type == 'input':
            var = ctrl.Antecedent(universe, name)
        else:
            var = ctrl.Consequent(universe, name)
        
        self.variables[name] = {
            'variable': var,
            'type': var_type,
            'universe': universe,
            'labels': labels,
            'range': universe_range
        }
        
        self.membership_functions[name] = {}
        
        print(f"✅ Created {var_type} variable '{name}' with range {universe_range}")
        print(f"   Labels: {', '.join(labels)}")
    
    def add_membership_function(self, var_name, label, mf_type, parameters):
        """
        Add a membership function to a variable
        
        Parameters:
        -----------
        var_name : str
            Variable name
        label : str
            Linguistic label
        mf_type : str
            Type of membership function
        parameters : list
            Parameters for the membership function
        """
        var_obj = self.variables[var_name]['variable']
        universe = self.variables[var_name]['universe']
        
        if mf_type == 'triangular':
            var_obj[label] = fuzz.trimf(universe, parameters)
        elif mf_type == 'trapezoidal':
            var_obj[label] = fuzz.trapmf(universe, parameters)
        elif mf_type == 'gaussian':
            var_obj[label] = fuzz.gaussmf(universe, parameters[0], parameters[1])
        elif mf_type == 'sigmoid':
            var_obj[label] = fuzz.sigmf(universe, parameters[0], parameters[1])
        
        self.membership_functions[var_name][label] = {
            'type': mf_type,
            'parameters': parameters
        }
        
        print(f"✅ Added {mf_type} membership function '{label}' to '{var_name}'")
    
    def add_rule(self, rule_text):
        """
        Add a fuzzy rule to the system
        
        Parameters:
        -----------
        rule_text : str
            Rule description for display
        rule : ctrl.Rule
            Fuzzy rule object
        """
        self.rules.append(rule_text)
        print(f"✅ Added rule: {rule_text}")
    
    def create_control_system(self, rules):
        """
        Create the fuzzy control system with rules
        
        Parameters:
        -----------
        rules : list
            List of ctrl.Rule objects
        """
        self.control_system = ctrl.ControlSystem(rules)
        self.simulation = ctrl.ControlSystemSimulation(self.control_system)
        print("✅ Fuzzy control system created successfully!")
    
    def calculate_output(self, inputs):
        """
        Calculate crisp output for given inputs
        
        Parameters:
        -----------
        inputs : dict
            Input values {variable_name: value}
            
        Returns:
        --------
        dict : Output values and analysis
        """
        if not self.simulation:
            raise ValueError("Control system not created yet!")
        
        # Set inputs
        for var_name, value in inputs.items():
            if var_name in self.variables and self.variables[var_name]['type'] == 'input':
                self.simulation.input[var_name] = value
        
        # Compute result
        self.simulation.compute()
        
        # Get outputs
        outputs = {}
        for var_name, var_info in self.variables.items():
            if var_info['type'] == 'output':
                outputs[var_name] = self.simulation.output[var_name]
        
        # Calculate membership degrees for inputs
        input_memberships = {}
        for var_name, value in inputs.items():
            if var_name in self.variables:
                var_obj = self.variables[var_name]['variable']
                input_memberships[var_name] = {}
                
                for label in self.variables[var_name]['labels']:
                    if hasattr(var_obj[label], 'mf'):
                        membership = fuzz.interp_membership(
                            self.variables[var_name]['universe'],
                            var_obj[label].mf,
                            value
                        )
                        input_memberships[var_name][label] = membership
        
        return {
            'inputs': inputs,
            'outputs': outputs,
            'input_memberships': input_memberships
        }
    
    def plot_membership_functions(self, var_name):
        """
        Plot membership functions for a variable
        """
        if var_name not in self.variables:
            print(f"Variable '{var_name}' not found!")
            return
        
        var_info = self.variables[var_name]
        var_obj = var_info['variable']
        universe = var_info['universe']
        
        plt.figure(figsize=(10, 6))
        colors = ['blue', 'green', 'red', 'orange', 'purple', 'brown']
        
        for i, label in enumerate(var_info['labels']):
            if hasattr(var_obj[label], 'mf'):
                color = colors[i % len(colors)]
                plt.plot(universe, var_obj[label].mf, color=color, linewidth=2.5, 
                        label=label, alpha=0.8)
                plt.fill_between(universe, 0, var_obj[label].mf, color=color, alpha=0.2)
        
        plt.xlim([universe.min(), universe.max()])
        plt.ylim([0, 1.05])
        plt.xlabel(f'{var_name} ({var_info["type"]})')
        plt.ylabel('Membership Degree')
        plt.title(f'Membership Functions for {var_name.title()}')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    def visualize_inference_process(self, inputs, outputs, input_memberships):
        """
        Create comprehensive visualization of the inference process
        """
        # Calculate number of subplots needed
        input_vars = [name for name, info in self.variables.items() if info['type'] == 'input']
        output_vars = [name for name, info in self.variables.items() if info['type'] == 'output']
        
        total_vars = len(input_vars) + len(output_vars)
        cols = min(3, total_vars)
        rows = (total_vars + cols - 1) // cols
        
        if rows < 2:
            rows = 2  # Ensure space for membership degrees plot
        
        fig = plt.figure(figsize=(15, 4 * rows))
        plot_idx = 1
        
        # Plot input variables
        for var_name in input_vars:
            ax = plt.subplot(rows, cols, plot_idx)
            var_info = self.variables[var_name]
            var_obj = var_info['variable']
            universe = var_info['universe']
            
            colors = ['blue', 'green', 'red', 'orange', 'purple']
            for i, label in enumerate(var_info['labels']):
                if hasattr(var_obj[label], 'mf'):
                    color = colors[i % len(colors)]
                    ax.plot(universe, var_obj[label].mf, color=color, linewidth=2, 
                           label=label, alpha=0.8)
                    ax.fill_between(universe, 0, var_obj[label].mf, color=color, alpha=0.2)
            
            # Mark input value
            ax.axvline(x=inputs[var_name], color='black', linestyle='--', linewidth=3, 
                      alpha=0.8, label=f'Input: {inputs[var_name]:.1f}')
            
            ax.set_xlim([universe.min(), universe.max()])
            ax.set_ylim([0, 1.05])
            ax.set_title(f'{var_name.title()} (Input)')
            ax.set_ylabel('Membership')
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
            
            plot_idx += 1
        
        # Plot output variables
        for var_name in output_vars:
            ax = plt.subplot(rows, cols, plot_idx)
            var_info = self.variables[var_name]
            var_obj = var_info['variable']
            universe = var_info['universe']
            
            colors = ['blue', 'green', 'red', 'orange', 'purple']
            for i, label in enumerate(var_info['labels']):
                if hasattr(var_obj[label], 'mf'):
                    color = colors[i % len(colors)]
                    ax.plot(universe, var_obj[label].mf, color=color, linewidth=2, 
                           label=label, alpha=0.8)
                    ax.fill_between(universe, 0, var_obj[label].mf, color=color, alpha=0.2)
            
            # Mark output value
            ax.axvline(x=outputs[var_name], color='red', linestyle='-', linewidth=4, 
                      alpha=0.8, label=f'Output: {outputs[var_name]:.2f}')
            
            ax.set_xlim([universe.min(), universe.max()])
            ax.set_ylim([0, 1.05])
            ax.set_title(f'{var_name.title()} (Output)')
            ax.set_ylabel('Membership')
            ax.set_xlabel('Universe')
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
            
            plot_idx += 1
        
        # Plot membership degrees bar chart
        if plot_idx <= rows * cols:
            ax = plt.subplot(rows, cols, plot_idx)
            categories = []
            values = []
            colors_bar = []
            
            for var_name, memberships in input_memberships.items():
                for label, value in memberships.items():
                    if value > 0.01:  # Only show significant memberships
                        categories.append(f'{var_name}\n{label}')
                        values.append(value)
                        colors_bar.append('#2E86AB' if value < 0.5 else '#A23B72')
            
            if categories:
                bars = ax.bar(categories, values, color=colors_bar, alpha=0.7)
                ax.set_title('Input Membership Degrees')
                ax.set_ylabel('Membership Degree')
                ax.tick_params(axis='x', rotation=45)
                
                # Add value labels on bars
                for bar, value in zip(bars, values):
                    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                           f'{value:.3f}', ha='center', va='bottom', fontsize=8)
        
        plt.suptitle('🔍 Complete Fuzzy Inference Process Visualization', 
                    fontsize=16, fontweight='bold', y=0.98)
        plt.tight_layout()
        plt.show()
    
    def create_summary_report(self, inputs, outputs, input_memberships):
        """
        Create a detailed summary report of the inference process
        """
        print("\n" + "="*60)
        print("📊 FUZZY INFERENCE SYSTEM - ANALYSIS REPORT")
        print("="*60)
        
        # Input Analysis
        print("\n🔹 INPUT ANALYSIS:")
        print("-" * 30)
        for var_name, value in inputs.items():
            print(f"📊 {var_name.title()}: {value:.2f}")
            
            if var_name in input_memberships:
                print("   Membership degrees:")
                for label, membership in input_memberships[var_name].items():
                    if membership > 0.01:
                        percentage = membership * 100
                        bar = "█" * int(percentage / 5)
                        print(f"     • {label}: {membership:.3f} ({percentage:.1f}%) {bar}")
            print()
        
        # Output Analysis
        print("🔹 OUTPUT ANALYSIS:")
        print("-" * 30)
        for var_name, value in outputs.items():
            print(f"🎯 {var_name.title()}: {value:.3f}")
            
            # Determine which output category this belongs to
            var_info = self.variables[var_name]
            var_obj = var_info['variable']
            
            print("   Output interpretation:")
            for label in var_info['labels']:
                if hasattr(var_obj[label], 'mf'):
                    membership = fuzz.interp_membership(
                        var_info['universe'], var_obj[label].mf, value
                    )
                    if membership > 0.01:
                        percentage = membership * 100
                        bar = "█" * int(percentage / 5)
                        print(f"     • {label}: {membership:.3f} ({percentage:.1f}%) {bar}")
        
        # System Summary
        print("\n🔹 SYSTEM SUMMARY:")
        print("-" * 30)
        print(f"📝 Total Variables: {len(self.variables)}")
        print(f"📝 Input Variables: {len([v for v in self.variables.values() if v['type'] == 'input'])}")
        print(f"📝 Output Variables: {len([v for v in self.variables.values() if v['type'] == 'output'])}")
        print(f"📝 Total Rules: {len(self.rules)}")
        
        # Active Rules (simplified)
        print("\n🔹 INFERENCE PROCESS:")
        print("-" * 30)
        print("✅ 1. Fuzzification: Input values converted to membership degrees")
        print("✅ 2. Rule Evaluation: Fuzzy rules applied to determine output")
        print("✅ 3. Aggregation: All rule outputs combined using fuzzy operators")
        print("✅ 4. Defuzzification: Fuzzy output converted to crisp value")
        
        print("\n" + "="*60)
        print("🎉 Analysis Complete!")
        print("="*60)

# Initialize the designer
designer = FuzzySystemDesigner()
print("\n🎛️ Fuzzy System Designer ready for use!")
print("\n📋 Quick Start Guide:")
print("   1. Use create_variable() to define inputs and outputs")
print("   2. Use add_membership_function() to design fuzzy sets")
print("   3. Use add_rule() and create_control_system() to build logic")
print("   4. Use calculate_output() to test your system")
print("   5. Use visualization methods to analyze results")

## 🚀 **Demo 1: Smart Home Temperature Control System**

Let's create a complete fuzzy system for smart home temperature control that considers:

### 📊 **System Design:**
- **Inputs**: Temperature (°F), Humidity (%), Time of Day (0-24h)
- **Output**: Fan Speed (0-100%)

### 🎯 **Linguistic Variables:**
- **Temperature**: Cold, Comfortable, Hot
- **Humidity**: Low, Medium, High  
- **Time**: Night, Day, Evening
- **Fan Speed**: Low, Medium, High

In [ ]:
# 🏠 DEMO 1: Smart Home Temperature Control System
print("🏠 Creating Smart Home Temperature Control System")
print("=" * 50)

# Step 1: Create Variables
print("\n📊 Step 1: Defining System Variables")

# Input variables
designer.create_variable('temperature', 'input', (60, 90), ['cold', 'comfortable', 'hot'])
designer.create_variable('humidity', 'input', (20, 80), ['low', 'medium', 'high'])
designer.create_variable('time_of_day', 'input', (0, 24), ['night', 'day', 'evening'])

# Output variable
designer.create_variable('fan_speed', 'output', (0, 100), ['low', 'medium', 'high'])

print("\n✅ All variables created successfully!")

In [ ]:
# Step 2: Design Membership Functions
print("\n🎛️ Step 2: Designing Membership Functions")
print("-" * 40)

# Temperature membership functions
designer.add_membership_function('temperature', 'cold', 'trapezoidal', [60, 60, 68, 72])
designer.add_membership_function('temperature', 'comfortable', 'triangular', [68, 75, 82])
designer.add_membership_function('temperature', 'hot', 'trapezoidal', [78, 82, 90, 90])

# Humidity membership functions
designer.add_membership_function('humidity', 'low', 'trapezoidal', [20, 20, 30, 40])
designer.add_membership_function('humidity', 'medium', 'triangular', [35, 50, 65])
designer.add_membership_function('humidity', 'high', 'trapezoidal', [60, 70, 80, 80])

# Time of day membership functions
designer.add_membership_function('time_of_day', 'night', 'trapezoidal', [0, 0, 6, 8])
designer.add_membership_function('time_of_day', 'day', 'trapezoidal', [7, 9, 17, 19])
designer.add_membership_function('time_of_day', 'evening', 'trapezoidal', [18, 20, 24, 24])

# Fan speed membership functions
designer.add_membership_function('fan_speed', 'low', 'triangular', [0, 0, 40])
designer.add_membership_function('fan_speed', 'medium', 'triangular', [20, 50, 80])
designer.add_membership_function('fan_speed', 'high', 'triangular', [60, 100, 100])

print("\n✅ All membership functions created!")

In [ ]:
# Step 3: Visualize Membership Functions
print("\n📈 Step 3: Visualizing Membership Functions")
print("-" * 40)

# Plot all variables
for var_name in ['temperature', 'humidity', 'time_of_day', 'fan_speed']:
    print(f"\n📊 Plotting {var_name}:")
    designer.plot_membership_functions(var_name)

In [ ]:
# Step 4: Create Fuzzy Rules
print("\n📋 Step 4: Creating Fuzzy Rules")
print("-" * 40)

# Get variable objects for rule creation
temp = designer.variables['temperature']['variable']
humidity = designer.variables['humidity']['variable']
time_day = designer.variables['time_of_day']['variable']
fan = designer.variables['fan_speed']['variable']

# Define comprehensive rule set
rules = [
    # Temperature-based rules
    ctrl.Rule(temp['cold'], fan['low']),
    ctrl.Rule(temp['hot'], fan['high']),
    
    # Humidity influence
    ctrl.Rule(humidity['high'] & temp['comfortable'], fan['medium']),
    ctrl.Rule(humidity['high'] & temp['hot'], fan['high']),
    ctrl.Rule(humidity['low'] & temp['comfortable'], fan['low']),
    
    # Time-based adjustments
    ctrl.Rule(time_day['night'] & temp['comfortable'], fan['low']),
    ctrl.Rule(time_day['day'] & temp['comfortable'] & humidity['medium'], fan['medium']),
    ctrl.Rule(time_day['evening'] & temp['comfortable'], fan['low']),
    
    # Complex combinations
    ctrl.Rule(temp['hot'] & humidity['high'], fan['high']),
    ctrl.Rule(temp['cold'] & humidity['low'], fan['low'])
]

# Add rule descriptions
rule_descriptions = [
    "IF temperature is cold THEN fan speed is low",
    "IF temperature is hot THEN fan speed is high",
    "IF humidity is high AND temperature is comfortable THEN fan speed is medium",
    "IF humidity is high AND temperature is hot THEN fan speed is high",
    "IF humidity is low AND temperature is comfortable THEN fan speed is low",
    "IF time is night AND temperature is comfortable THEN fan speed is low",
    "IF time is day AND temperature is comfortable AND humidity is medium THEN fan speed is medium",
    "IF time is evening AND temperature is comfortable THEN fan speed is low",
    "IF temperature is hot AND humidity is high THEN fan speed is high",
    "IF temperature is cold AND humidity is low THEN fan speed is low"
]

for desc in rule_descriptions:
    designer.add_rule(desc)

# Create control system
designer.create_control_system(rules)

print(f"\n✅ Fuzzy control system created with {len(rules)} rules!")

In [ ]:
# Step 5: Interactive Testing Interface
print("\n🧪 Step 5: Interactive System Testing")
print("-" * 40)

def test_fuzzy_system(temperature_val=75, humidity_val=50, time_val=14):
    """
    Interactive function to test the fuzzy system
    """
    # Clear previous output
    clear_output(wait=True)
    
    # Create inputs
    inputs = {
        'temperature': temperature_val,
        'humidity': humidity_val,
        'time_of_day': time_val
    }
    
    # Calculate output
    result = designer.calculate_output(inputs)
    
    # Display results
    print("🏠 Smart Home Temperature Control System - Live Test")
    print("=" * 55)
    
    print(f"\n📊 Current Conditions:")
    print(f"   🌡️  Temperature: {temperature_val}°F")
    print(f"   💧 Humidity: {humidity_val}%")
    print(f"   🕐 Time: {time_val}:00")
    
    print(f"\n🎯 System Response:")
    fan_speed = result['outputs']['fan_speed']
    print(f"   💨 Fan Speed: {fan_speed:.1f}%")
    
    # Interpretation
    if fan_speed <= 30:
        interpretation = "Low speed - Minimal cooling needed"
        energy = "Energy Efficient ⭐⭐⭐"
    elif fan_speed <= 70:
        interpretation = "Medium speed - Moderate cooling"
        energy = "Balanced ⭐⭐"
    else:
        interpretation = "High speed - Maximum cooling"
        energy = "High consumption ⭐"
    
    print(f"   📝 Interpretation: {interpretation}")
    print(f"   ⚡ Energy Usage: {energy}")
    
    # Visualize the complete inference process
    designer.visualize_inference_process(
        result['inputs'], 
        result['outputs'], 
        result['input_memberships']
    )
    
    # Create detailed report
    designer.create_summary_report(
        result['inputs'], 
        result['outputs'], 
        result['input_memberships']
    )

# Create interactive widgets
temp_slider = widgets.FloatSlider(
    value=75, min=60, max=90, step=0.5,
    description='Temperature (°F):',
    style={'description_width': 'initial'}
)

humidity_slider = widgets.FloatSlider(
    value=50, min=20, max=80, step=1,
    description='Humidity (%):',
    style={'description_width': 'initial'}
)

time_slider = widgets.FloatSlider(
    value=14, min=0, max=24, step=0.5,
    description='Time (24h):',
    style={'description_width': 'initial'}
)

# Create interactive interface
print("🎛️ Interactive Fuzzy System Tester")
print("Adjust the sliders below to test different conditions:")
print()

interactive_plot = interactive(
    test_fuzzy_system,
    temperature_val=temp_slider,
    humidity_val=humidity_slider,
    time_val=time_slider
)

display(interactive_plot)

## 🍽️ **Demo 2: Restaurant Service Rating System**

Let's create another fuzzy system for restaurant service evaluation:

### 📊 **System Design:**
- **Inputs**: Service Quality (1-10), Food Quality (1-10), Price Level (1-10)
- **Output**: Overall Rating (1-10)

This demonstrates a different application domain with different membership function shapes.

In [ ]:
# 🍽️ DEMO 2: Restaurant Service Rating System
print("🍽️ Creating Restaurant Service Rating System")
print("=" * 50)

# Create a new designer instance for the restaurant system
restaurant_designer = FuzzySystemDesigner()

# Step 1: Define Variables
print("\n📊 Step 1: Defining Restaurant Rating Variables")

# Input variables
restaurant_designer.create_variable('service', 'input', (1, 10), ['poor', 'average', 'excellent'])
restaurant_designer.create_variable('food', 'input', (1, 10), ['bad', 'good', 'outstanding'])
restaurant_designer.create_variable('price', 'input', (1, 10), ['cheap', 'reasonable', 'expensive'])

# Output variable
restaurant_designer.create_variable('rating', 'output', (1, 10), ['poor', 'average', 'excellent'])

# Step 2: Create Membership Functions with different shapes
print("\n🎛️ Step 2: Designing Membership Functions")

# Service quality (using Gaussian for smoother transitions)
restaurant_designer.add_membership_function('service', 'poor', 'gaussian', [3, 1.5])
restaurant_designer.add_membership_function('service', 'average', 'gaussian', [5.5, 1.5])
restaurant_designer.add_membership_function('service', 'excellent', 'gaussian', [8, 1.5])

# Food quality (using triangular)
restaurant_designer.add_membership_function('food', 'bad', 'triangular', [1, 1, 4])
restaurant_designer.add_membership_function('food', 'good', 'triangular', [3, 6, 9])
restaurant_designer.add_membership_function('food', 'outstanding', 'triangular', [7, 10, 10])

# Price level (using trapezoidal)
restaurant_designer.add_membership_function('price', 'cheap', 'trapezoidal', [1, 1, 3, 4])
restaurant_designer.add_membership_function('price', 'reasonable', 'trapezoidal', [3, 4, 6, 7])
restaurant_designer.add_membership_function('price', 'expensive', 'trapezoidal', [6, 7, 10, 10])

# Rating output (using triangular)
restaurant_designer.add_membership_function('rating', 'poor', 'triangular', [1, 1, 5])
restaurant_designer.add_membership_function('rating', 'average', 'triangular', [3, 5.5, 8])
restaurant_designer.add_membership_function('rating', 'excellent', 'triangular', [6, 10, 10])

print("\n✅ Restaurant system variables and membership functions created!")

In [ ]:
# Step 3: Create Restaurant Rating Rules
print("\n📋 Step 3: Creating Restaurant Rating Rules")
print("-" * 40)

# Get variable objects
service = restaurant_designer.variables['service']['variable']
food = restaurant_designer.variables['food']['variable']
price = restaurant_designer.variables['price']['variable']
rating = restaurant_designer.variables['rating']['variable']

# Define restaurant rating rules
restaurant_rules = [
    # Excellent combinations
    ctrl.Rule(service['excellent'] & food['outstanding'], rating['excellent']),
    ctrl.Rule(service['excellent'] & food['good'] & price['reasonable'], rating['excellent']),
    
    # Good combinations
    ctrl.Rule(service['average'] & food['outstanding'], rating['average']),
    ctrl.Rule(service['excellent'] & food['good'], rating['average']),
    ctrl.Rule(service['average'] & food['good'] & price['cheap'], rating['average']),
    
    # Poor combinations
    ctrl.Rule(service['poor'], rating['poor']),
    ctrl.Rule(food['bad'], rating['poor']),
    ctrl.Rule(service['poor'] | food['bad'], rating['poor']),
    
    # Price impact
    ctrl.Rule(price['expensive'] & service['average'] & food['good'], rating['poor']),
    ctrl.Rule(price['cheap'] & service['average'] & food['good'], rating['average'])
]

# Rule descriptions
restaurant_rule_descriptions = [
    "IF service is excellent AND food is outstanding THEN rating is excellent",
    "IF service is excellent AND food is good AND price is reasonable THEN rating is excellent",
    "IF service is average AND food is outstanding THEN rating is average",
    "IF service is excellent AND food is good THEN rating is average",
    "IF service is average AND food is good AND price is cheap THEN rating is average",
    "IF service is poor THEN rating is poor",
    "IF food is bad THEN rating is poor",
    "IF service is poor OR food is bad THEN rating is poor",
    "IF price is expensive AND service is average AND food is good THEN rating is poor",
    "IF price is cheap AND service is average AND food is good THEN rating is average"
]

for desc in restaurant_rule_descriptions:
    restaurant_designer.add_rule(desc)

# Create control system
restaurant_designer.create_control_system(restaurant_rules)

print(f"\n✅ Restaurant rating system created with {len(restaurant_rules)} rules!")

In [ ]:
# Step 4: Interactive Restaurant Rating Tester
print("\n🧪 Step 4: Interactive Restaurant Rating System")
print("-" * 40)

def test_restaurant_rating(service_val=7, food_val=8, price_val=5):
    """
    Interactive function to test the restaurant rating system
    """
    clear_output(wait=True)
    
    # Create inputs
    inputs = {
        'service': service_val,
        'food': food_val,
        'price': price_val
    }
    
    # Calculate output
    result = restaurant_designer.calculate_output(inputs)
    
    # Display results
    print("🍽️ Restaurant Service Rating System - Live Evaluation")
    print("=" * 55)
    
    print(f"\n📊 Restaurant Experience:")
    print(f"   👨‍💼 Service Quality: {service_val}/10")
    print(f"   🍕 Food Quality: {food_val}/10")
    print(f"   💰 Price Level: {price_val}/10")
    
    print(f"\n🎯 Overall Rating:")
    overall_rating = result['outputs']['rating']
    print(f"   ⭐ Rating: {overall_rating:.2f}/10")
    
    # Rating interpretation
    if overall_rating <= 3.5:
        interpretation = "Poor - Would not recommend"
        stars = "⭐"
    elif overall_rating <= 6.5:
        interpretation = "Average - Decent experience"
        stars = "⭐⭐⭐"
    else:
        interpretation = "Excellent - Highly recommended!"
        stars = "⭐⭐⭐⭐⭐"
    
    print(f"   📝 Assessment: {interpretation}")
    print(f"   🌟 Stars: {stars}")
    
    # Visualize the inference process
    restaurant_designer.visualize_inference_process(
        result['inputs'], 
        result['outputs'], 
        result['input_memberships']
    )
    
    # Create detailed report
    restaurant_designer.create_summary_report(
        result['inputs'], 
        result['outputs'], 
        result['input_memberships']
    )

# Create sliders for restaurant rating
service_slider = widgets.FloatSlider(
    value=7, min=1, max=10, step=0.1,
    description='Service Quality:',
    style={'description_width': 'initial'}
)

food_slider = widgets.FloatSlider(
    value=8, min=1, max=10, step=0.1,
    description='Food Quality:',
    style={'description_width': 'initial'}
)

price_slider = widgets.FloatSlider(
    value=5, min=1, max=10, step=0.1,
    description='Price Level:',
    style={'description_width': 'initial'}
)

print("🎛️ Interactive Restaurant Rating Tester")
print("Rate different aspects to see the overall restaurant rating:")
print()

restaurant_interactive = interactive(
    test_restaurant_rating,
    service_val=service_slider,
    food_val=food_slider,
    price_val=price_slider
)

display(restaurant_interactive)

## 🎯 **Quick System Builder**

Use the tools below to quickly create your own fuzzy system:

### 🛠️ **Custom System Builder Interface:**
- Define your own variables and membership functions
- Create custom rules
- Test with your own data
- Export/import system configurations

In [ ]:
# 🛠️ CUSTOM FUZZY SYSTEM BUILDER
print("🛠️ Custom Fuzzy System Builder")
print("=" * 40)

class QuickFuzzyBuilder:
    """
    Simplified interface for quick fuzzy system creation
    """
    
    def __init__(self):
        self.designer = FuzzySystemDesigner()
        
    def quick_setup(self, system_name, input_configs, output_config, rules_text):
        """
        Quick setup for a fuzzy system
        
        Parameters:
        -----------
        system_name : str
            Name of the system
        input_configs : list of dict
            Input variable configurations
        output_config : dict
            Output variable configuration
        rules_text : list
            Human-readable rule descriptions
        """
        print(f"🔧 Building '{system_name}' fuzzy system...")
        
        # Create input variables
        for config in input_configs:
            self.designer.create_variable(
                config['name'], 'input', 
                config['range'], config['labels']
            )
            
            # Add membership functions
            for mf_config in config['membership_functions']:
                self.designer.add_membership_function(
                    config['name'], mf_config['label'],
                    mf_config['type'], mf_config['params']
                )
        
        # Create output variable
        self.designer.create_variable(
            output_config['name'], 'output',
            output_config['range'], output_config['labels']
        )
        
        # Add output membership functions
        for mf_config in output_config['membership_functions']:
            self.designer.add_membership_function(
                output_config['name'], mf_config['label'],
                mf_config['type'], mf_config['params']
            )
        
        print(f"✅ System '{system_name}' created successfully!")
        print(f"📊 Variables: {len(input_configs)} inputs, 1 output")
        
        return self.designer

# Example: Investment Risk Assessment System
print("\n💰 Example: Investment Risk Assessment System")
print("-" * 50)

# Create quick builder
quick_builder = QuickFuzzyBuilder()

# Define investment system configuration
input_configs = [
    {
        'name': 'market_volatility',
        'range': (0, 100),
        'labels': ['low', 'medium', 'high'],
        'membership_functions': [
            {'label': 'low', 'type': 'triangular', 'params': [0, 0, 40]},
            {'label': 'medium', 'type': 'triangular', 'params': [20, 50, 80]},
            {'label': 'high', 'type': 'triangular', 'params': [60, 100, 100]}
        ]
    },
    {
        'name': 'company_performance',
        'range': (0, 10),
        'labels': ['poor', 'average', 'excellent'],
        'membership_functions': [
            {'label': 'poor', 'type': 'triangular', 'params': [0, 0, 4]},
            {'label': 'average', 'type': 'triangular', 'params': [2, 5, 8]},
            {'label': 'excellent', 'type': 'triangular', 'params': [6, 10, 10]}
        ]
    }
]

output_config = {
    'name': 'risk_level',
    'range': (0, 100),
    'labels': ['low_risk', 'medium_risk', 'high_risk'],
    'membership_functions': [
        {'label': 'low_risk', 'type': 'triangular', 'params': [0, 0, 40]},
        {'label': 'medium_risk', 'type': 'triangular', 'params': [20, 50, 80]},
        {'label': 'high_risk', 'type': 'triangular', 'params': [60, 100, 100]}
    ]
}

# Create the system
investment_designer = quick_builder.quick_setup(
    "Investment Risk Assessment",
    input_configs,
    output_config,
    []
)

print("\n🎯 Investment Risk Assessment System ready for rules and testing!")
print("\n📋 System Summary:")
print("   • Market Volatility (0-100): Low, Medium, High")
print("   • Company Performance (0-10): Poor, Average, Excellent")
print("   • Risk Level (0-100): Low Risk, Medium Risk, High Risk")

# Display membership functions
print("\n📊 Visualizing Investment System Variables:")
for var_name in ['market_volatility', 'company_performance', 'risk_level']:
    investment_designer.plot_membership_functions(var_name)

## 🎓 **Tool Summary and Advanced Features**

### ✅ **What You've Built:**

1. **🏠 Smart Home System**: Temperature control with 3 inputs
2. **🍽️ Restaurant Rating**: Service evaluation with different MF types
3. **💰 Investment Assessment**: Quick builder demonstration

### 🛠️ **Tool Capabilities:**

- **Variable Creation**: Define inputs/outputs with custom ranges
- **Membership Function Design**: 4 types with parameter control
- **Rule System**: Logical IF-THEN rule creation
- **Real-time Testing**: Interactive sliders for live testing
- **Complete Visualization**: Step-by-step inference process
- **Detailed Analysis**: Comprehensive reports and interpretations

### 🚀 **Next Steps:**

- **Extend Rules**: Add more complex rule combinations
- **Optimize Parameters**: Fine-tune membership function shapes
- **Save/Load Systems**: Export configurations for reuse
- **Integration**: Connect to real sensors and actuators
- **Performance Analysis**: Test with real-world datasets

**You now have a complete fuzzy logic design toolkit!** 🎉